In [ ]:
!mkdir -p /content/_MusicProject
!unzip -q features_only.zip -d /content/_MusicProject
!ls -R /content/_MusicProject

/content/_MusicProject:
data

/content/_MusicProject/data:
features

/content/_MusicProject/data/features:
general  study

/content/_MusicProject/data/features/general:
'1st Contact - Duty Free.npz'
'99 Tales - Baby Out Of Jail.npz'
'A. Cooper - Aggression.npz'
'A. Cooper - Support.npz'
'Alaclair Ensemble - LICORNES.npz'
'Allister Thompson - The Old Miner.npz'
'Andrea - Work the Middle (Real Remix).npz'
'An Eagle In Your Mind - Cave of the Darling.npz'
'Austin Leonard Jones - Florida.npz'
'batchbug-snowflake(chosic.com).npz'
'Beat Mekanik - Harley Davidson.npz'
'BJ Block & Dawn Pemberton - Turn It Around.npz'
'Breuss Arrizabalaga Quintet - Tsurugi.npz'
'BrokeMC - broke Pimpin.npz'
'Bryan Mathys - Hard Miles.npz'
'Carlos Javed - Tiger - Roar of Destiny.npz'
'Carroll - Goat.npz'
'Clan Destined - Never All Ways (clean).npz'
'Cletus Got Shot - Charlie Jones.npz'
'Combo - The Lake.npz'
"Cullah - I'll Never Understand.npz"
'DARKNESS(chosic.com).npz'
'Dasnaty - I gat you.npz'
'Dead Times - Sl

In [ ]:
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

root = Path("/content/_MusicProject")
features_study_dir = root / "data" / "features" / "study"
features_general_dir = root / "data" / "features" / "general"

X = []
y = []
paths = []

for npz_path in features_study_dir.glob("*.npz"):
    d = np.load(npz_path, allow_pickle=True)
    X.append(d["features"])
    y.append(int(d["label"]))
    paths.append(str(d["path"]))

for npz_path in features_general_dir.glob("*.npz"):
    d = np.load(npz_path, allow_pickle=True)
    X.append(d["features"])
    y.append(int(d["label"]))
    paths.append(str(d["path"]))

X = np.stack(X)
y = np.array(y)
paths = np.array(paths)

print("[INFO] X shape:", X.shape)
print("[INFO] y shape:", y.shape)
print("[INFO] num study:", np.sum(y == 1))
print("[INFO] num general:", np.sum(y == 0))

[INFO] X shape: (236, 44)
[INFO] y shape: (236,)
[INFO] num study: 122
[INFO] num general: 114


In [ ]:
X_train, X_temp, y_train, y_temp, paths_train, paths_temp = train_test_split(
    X, y, paths, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test, paths_val, paths_test = train_test_split(
    X_temp, y_temp, paths_temp, test_size=0.5, random_state=42, stratify=y_temp
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("[INFO] Train size:", X_train_scaled.shape[0])
print("[INFO] Val size:", X_val_scaled.shape[0])
print("[INFO] Test size:", X_test_scaled.shape[0])

[INFO] Train size: 165
[INFO] Val size: 35
[INFO] Test size: 36


In [ ]:
input_dim = X_train_scaled.shape[1]

model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.AUC(name="auc")
    ]
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=80,
    batch_size=16,
    callbacks=[early_stop],
    verbose=2
)

Epoch 1/80
11/11 - 7s - 632ms/step - accuracy: 0.6061 - auc: 0.6605 - loss: 0.6414 - val_accuracy: 0.7429 - val_auc: 0.7843 - val_loss: 0.6020
Epoch 2/80
11/11 - 0s - 10ms/step - accuracy: 0.8242 - auc: 0.9073 - loss: 0.4535 - val_accuracy: 0.8857 - val_auc: 0.9085 - val_loss: 0.4892
Epoch 3/80
11/11 - 0s - 9ms/step - accuracy: 0.9152 - auc: 0.9582 - loss: 0.3420 - val_accuracy: 0.9143 - val_auc: 0.9314 - val_loss: 0.4222
Epoch 4/80
11/11 - 0s - 9ms/step - accuracy: 0.9091 - auc: 0.9569 - loss: 0.2909 - val_accuracy: 0.9429 - val_auc: 0.9314 - val_loss: 0.3848
Epoch 5/80
11/11 - 0s - 9ms/step - accuracy: 0.9152 - auc: 0.9671 - loss: 0.2502 - val_accuracy: 0.9429 - val_auc: 0.9314 - val_loss: 0.3667
Epoch 6/80
11/11 - 0s - 9ms/step - accuracy: 0.9394 - auc: 0.9775 - loss: 0.2188 - val_accuracy: 0.9429 - val_auc: 0.9379 - val_loss: 0.3528
Epoch 7/80
11/11 - 0s - 9ms/step - accuracy: 0.9333 - auc: 0.9857 - loss: 0.1736 - val_accuracy: 0.9143 - val_auc: 0.9395 - val_loss: 0.3438
Epoch 8/80

In [ ]:
y_probs = model.predict(X_test_scaled).ravel()
y_pred = (y_probs >= 0.5).astype(int)

test_acc = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_probs)

print("[TEST] Accuracy:", round(test_acc, 3))
print("[TEST] ROC-AUC:", round(test_auc, 3))
print("\n[TEST] Classification report:\n")
print(classification_report(y_test, y_pred))

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 365ms/step
[TEST] Accuracy: 0.917
[TEST] ROC-AUC: 0.985

[TEST] Classification report:

              precision    recall  f1-score   support

           0       0.85      1.00      0.92        17
           1       1.00      0.84      0.91        19

    accuracy                           0.92        36
   macro avg       0.93      0.92      0.92        36
weighted avg       0.93      0.92      0.92        36



In [ ]:
models_dir = root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "focus_mlp_tf.keras"
scaler_path = models_dir / "focus_scaler_mlp_tf.npy"

model.save(str(model_path))
np.save(scaler_path, {"mean": scaler.mean_, "scale": scaler.scale_})

print("[INFO] Saved:")
print(model_path)
print(scaler_path)

[INFO] Saved:
/content/_MusicProject/models/focus_mlp_tf.keras
/content/_MusicProject/models/focus_scaler_mlp_tf.npy


In [ ]:
from google.colab import files
from pathlib import Path

root = Path("/content/_MusicProject")
models_dir = root / "models"

model_path = models_dir / "focus_mlp_tf.keras"
scaler_path = models_dir / "focus_scaler_mlp_tf.npy"

print("Downloading:", model_path)
files.download(str(model_path))

print("Downloading:", scaler_path)
files.download(str(scaler_path))


Downloading: /content/_MusicProject/models/focus_mlp_tf.keras


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: /content/_MusicProject/models/focus_scaler_mlp_tf.npy


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>